# reranker_colab.ipynb — 리랭커(BAAI/bge-reranker-v2-m3)를 코랩 런타임에서 돌리기 위한 노트북

배경: 로컬 컴퓨터(RAM 7.4GB)에서는 bge-reranker-v2-m3(568M 파라미터)를 로드하려고 하면
세그멘테이션 폴트로 프로세스가 죽는다(2026-08-13 실측 확인) — 임베딩(e5-small/e5-large)은
디스크 공간을 확보한 뒤 정상 동작했지만, 리랭커는 모델 자체가 더 커서 로컬 RAM으로는
부족한 것으로 보인다.

이 노트북은 VSCode의 Colab 확장으로 이 파일에 연결해서, 무거운 모델(리랭커·임베딩)만
코랩의 더 넉넉한 런타임에서 실행하기 위한 것이다. `agent/` 패키지 코드는 건드리지 않는다 —
같은 코드를 어디서 실행하느냐만 다르게 하는 것이 목표(로컬 코드와 갈라지지 않게).

## 사용법
1. VSCode에서 이 노트북 열기 → 우측 상단 "커널 선택" → "Colab" → "New Colab Server" → 구글 계정 로그인
2. 아래 셀을 순서대로 실행해서 (1) 연결 확인 → (2) 리랭커 로드 확인까지 검증
3. 여기까지 되면, 다음 단계로 실제 배치 파이프라인과 연동하는 방법을 이어서 설계한다

## 1. 연결 확인 — 이게 코랩에서 도는지, 로컬에서 도는지부터 확인

In [ ]:
import platform, os
print("플랫폼:", platform.platform())
print("코랩 여부(google.colab 모듈 있는지):", end=" ")
try:
    import google.colab  # noqa: F401
    print("코랩에서 실행 중")
except ImportError:
    print("코랩 아님 — 로컬에서 도는 중 (커널 선택이 안 된 상태일 수 있음)")

# 메모리 확인 (코랩이면 보통 12GB+, Pro면 더 많음)
!cat /proc/meminfo 2>/dev/null | head -3 || echo "(Linux 환경 아님 — Windows 로컬일 가능성)"

## 2. 리랭커(bge-reranker-v2-m3) 로드 테스트
로컬에서 세그멘테이션 폴트로 죽었던 바로 그 모델. 여기서 정상 로드되는지 확인.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import time
from sentence_transformers import CrossEncoder

t0 = time.time()
model = CrossEncoder("BAAI/bge-reranker-v2-m3")
print(f"로드 성공! 소요: {time.time() - t0:.1f}초")

score = model.predict([("소매 판매가 늘었다", "소매판매액지수")])
print("테스트 점수:", score)

## 3. 실제 파이프라인 연동

로컬에서 `python -m agent.pipeline.export_for_rerank --csv --csv-n 30`을 실행하면
`data/rerank_pending.json`이 생긴다 — 1~3단계(리랭킹 직전)까지 돌린 claim별 후보 목록.

이 파일을 아래 셀에서 업로드하면, 여기서 리랭커로 점수를 매기고 `rerank_results.json`을
다시 다운로드해준다. 그걸 로컬 `data/` 폴더에 넣고
`python -m agent.pipeline.resume_after_rerank`를 실행하면 4~8단계가 마저 진행된다.

점수 계산 방식(시그모이드 + 검증 보너스)은 `agent/mapping/reranker.py`의 `rerank()`
함수랑 똑같이 맞춰뒀다 — 로컬에서 리랭커가 직접 돌 때랑 결과가 갈라지지 않게.

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

# 아래 SRC 경로를 본인 드라이브에 업로드한 rerank_pending.json 실제 경로로 바꾸세요.
# (구글 드라이브 웹사이트나 드라이브 앱에서 이 파일을 "내 드라이브" 최상위에 끌어다 놓으면
# 기본값 그대로 써도 됩니다.)
SRC = "/content/drive/MyDrive/rerank_pending.json"
pending_filename = "rerank_pending.json"
shutil.copy(SRC, pending_filename)
print("복사됨:", pending_filename)

In [ ]:
import json

with open(pending_filename, encoding="utf-8") as f:
    pending = json.load(f)

print(f"리랭킹 대상 claim {len(pending['items'])}건, 카탈로그 {len(pending['catalog'])}개 표")

### 3-1. 임베딩 매칭 (로컬에서 세그폴트 나던 부분을 여기서 대신 실행)

`pending["catalog"]`에 카탈로그 전체(표별 embedding_text)가 들어있다 — 이걸로 표
임베딩을 만들고, claim마다 코사인 유사도로 후보를 찾은 뒤 keyword_candidates랑 합친다.
합치는 규칙은 `agent/mapping/reranker.py`의 `_merge_candidates()`와 동일: keyword_search가
찾은 표는 그대로 두고, 임베딩만으로 찾은 표는 "(embedding-only, unverified)"로 표시.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("intfloat/multilingual-e5-large")

catalog = pending["catalog"]
table_ids = list(catalog.keys())
passage_texts = ["passage: " + catalog[tid]["embedding_text"] for tid in table_ids]

print(f"카탈로그 {len(table_ids)}개 표 임베딩 생성 중...")
catalog_vecs = embed_model.encode(passage_texts, convert_to_numpy=True, normalize_embeddings=True)
print("완료")

EMBEDDING_TOP_K = 5


def embedding_candidates_for(claim_sentence):
    query_vec = embed_model.encode(
        ["query: " + claim_sentence], convert_to_numpy=True, normalize_embeddings=True
    )[0]
    sims = catalog_vecs @ query_vec
    top_idx = np.argsort(-sims)[:EMBEDDING_TOP_K]
    return [
        {
            "table_id": table_ids[i],
            "table_name": catalog[table_ids[i]]["table_name"],
            "score": float(sims[i]),
            "required_slots": [],
            "source_meta": "embedding_search model=intfloat/multilingual-e5-large",
        }
        for i in top_idx
    ]


def merge_candidates(keyword_cands, embedding_cands):
    # agent/mapping/reranker.py의 _merge_candidates()와 동일한 규칙: keyword_search가
    # 찾은 표는 그대로 신뢰, 임베딩만으로 찾은 표는 "(embedding-only, unverified)"로 표시.
    merged = {}
    for c in keyword_cands:
        merged[c["table_id"]] = c
    for c in embedding_cands:
        if c["table_id"] not in merged:
            merged[c["table_id"]] = {**c, "source_meta": f"{c['source_meta']} (embedding-only, unverified)"}
        else:
            existing = merged[c["table_id"]]
            merged[c["table_id"]] = {**existing, "source_meta": f"{existing['source_meta']} | {c['source_meta']}"}
    return list(merged.values())


for i, item in enumerate(pending["items"]):
    emb_cands = embedding_candidates_for(item["claim"]["sentence"])
    item["merged_candidates"] = merge_candidates(item["keyword_candidates"], emb_cands)
    if i % 10 == 0:
        print(f"임베딩 매칭 진행: {i}/{len(pending['items'])}")

print("임베딩 매칭 + 병합 완료")

In [ ]:
import math


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))


# agent/mapping/reranker.py의 _VERIFIED_BONUS와 반드시 같은 값으로 유지 — 로컬 경로랑
# 코랩 경로의 점수 계산 방식이 갈라지면 안 됨.
VERIFIED_BONUS = 0.05

output_items = []
for i, item in enumerate(pending["items"]):
    claim_sentence = item["claim"]["sentence"]
    candidates = item["merged_candidates"]
    if not candidates:
        output_items.append({"item_id": item["item_id"], "candidates": []})
        continue
    docs = [catalog[c["table_id"]]["embedding_text"] for c in candidates]

    raw_scores = model.predict([(claim_sentence, d) for d in docs])

    reranked = []
    for c, raw in zip(candidates, raw_scores):
        adjusted = sigmoid(float(raw))
        if "unverified" not in (c.get("source_meta") or ""):
            adjusted = min(adjusted + VERIFIED_BONUS, 1.0)
        reranked.append({**c, "score": adjusted, "raw_rerank_score": float(raw)})
    reranked.sort(key=lambda c: c["score"], reverse=True)

    output_items.append({"item_id": item["item_id"], "candidates": reranked[:5]})
    if i % 10 == 0:
        print(f"리랭킹 진행: {i}/{len(pending['items'])}")

print("리랭킹 완료:", len(output_items), "건")

In [ ]:
with open("rerank_results.json", "w", encoding="utf-8") as f:
    json.dump({"items": output_items}, f, ensure_ascii=False, indent=2)

files.download("rerank_results.json")
print("rerank_results.json 다운로드 완료 — data/ 폴더에 옮기고 resume_after_rerank.py 실행하세요.")